In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pickle

: 

In [ ]:
df = pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")

# Explainatory Data Analysis (EDA)

In [ ]:
df.head(10)

In [ ]:
# Shape of the dataset
df.shape

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Attrition', data=df, palette='Set2')
plt.title("Employee Attrition Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Gender', hue='Attrition', data=df, palette='Set1')
plt.title("Attrition by Gender")
plt.show()

In [ ]:
plt.figure(figsize=(7,5))
sns.countplot(x='Department', hue='Attrition', data=df, palette='Set2')
plt.title("Attrition by Department")
plt.xticks(rotation=30)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data=df, x='Age', hue='Attrition', kde=True, multiple='stack')
plt.title("Attrition by Age Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='Attrition', y='MonthlyIncome', data=df, palette='Set3')
plt.title("Monthly Income vs Attrition")
plt.show()

In [ ]:
plt.figure(figsize=(12,8))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title("Correlation Heatmap of Numerical Features")
plt.show()

In [ ]:
# Attrition distribution
df['Attrition'].value_counts()

In [ ]:
# Identify feature types
target_column = 'Attrition'
categorical_features = []
numerical_features = []

for column in df.columns:
    if column == target_column:
        continue
    elif df[column].dtype == 'object' or df[column].nunique() < 10:
        categorical_features.append(column)
    else:
        numerical_features.append(column)

print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"Numerical features ({len(numerical_features)}): {numerical_features}")

# Detect outliers using IQR
outlier_indices = set()
for feature in numerical_features:
    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    feature_outliers = df[(df[feature] < lower_bound) | (df[feature] > upper_bound)].index
    outlier_indices.update(feature_outliers)
    print(f"{feature}: {len(feature_outliers)} outliers detected")
print(f"Total unique outlier records: {len(outlier_indices)}")

In [ ]:
# Detect outliers using IQR
outlier_indices = set()
for feature in numerical_features:
    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    feature_outliers = df[(df[feature] < lower_bound) | (df[feature] > upper_bound)].index
    outlier_indices.update(feature_outliers)
    print(f"{feature}: {len(feature_outliers)} outliers detected")
print(f"Total unique outlier records: {len(outlier_indices)}")

# Preprocessing

In [ ]:
# Create a clean dataset by removing outlier records
df_clean = df.drop(index=outlier_indices)

In [ ]:
# Reset index if needed
df_clean = df_clean.reset_index(drop=True)

# Verify outliers are removed by checking a few features
print("\nVerification - outliers remaining after cleaning:")
for feature in ['MonthlyIncome', 'NumCompaniesWorked', 'TotalWorkingYears']:
    if feature in numerical_features:
        Q1 = df_clean[feature].quantile(0.25)
        Q3 = df_clean[feature].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        remaining_outliers = df_clean[(df_clean[feature] < lower_bound) | (df_clean[feature] > upper_bound)]
        print(f"{feature}: {len(remaining_outliers)} outliers remaining")

In [ ]:
# Attrition distribution
df_clean['Attrition'].value_counts()

In [ ]:
import pandas as pd
from sklearn.utils import resample

df_no = df_clean[df_clean['Attrition'] == 'No']
df_yes = df_clean[df_clean['Attrition'] == 'Yes']

# Upsample minority class
df_yes_upsampled = resample(df_yes,
                            replace=True,          # sample with replacement
                            n_samples=len(df_no),  # match number of majority class
                            random_state=42)       # reproducible results

# Combine majority and upsampled minority
df_balanced = pd.concat([df_no, df_yes_upsampled])

# Shuffle the dataframe
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanced['Attrition'].value_counts())


In [ ]:
# Drop useless columns
df_balanced.drop(['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours'], axis="columns", inplace=True)

# 1. Professionally select all text/categorical columns
categorical_col = df_balanced.select_dtypes(include=['object']).columns.tolist()

# 2. Safely remove 'Attrition' from this list since we handle it separately
if 'Attrition' in categorical_col:
    categorical_col.remove('Attrition')

print(f"Columns to encode: {categorical_col}")

# 3. Convert Attrition to numbers (0 for No, 1 for Yes)
df_balanced['Attrition'] = df_balanced['Attrition'].astype("category").cat.codes

# 4. Apply LabelEncoder to the rest of the text columns
label_encoders = {}  # dictionary to store encoders

for column in categorical_col:
    le = LabelEncoder()
    df_balanced[column] = le.fit_transform(df_balanced[column])
    label_encoders[column] = le  # save encoder for this column

# Save all label encoders to a file
with open('label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

In [ ]:
X = df_balanced.drop('Attrition', axis=1)
y = df_balanced.Attrition

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
X_train.isnull().sum()

In [ ]:
# Save datasets
with open('X_train.pkl', 'wb') as f:
    pickle.dump(X_train, f)

with open('X_test.pkl', 'wb') as f:
    pickle.dump(X_test, f)

with open('y_train.pkl', 'wb') as f:
    pickle.dump(y_train, f)

with open('y_test.pkl', 'wb') as f:
    pickle.dump(y_test, f)